<a href="https://colab.research.google.com/github/ablasve/Mini-Proyecto-Asistente-Multimodal-de-Salud/blob/main/ProyectoAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Mini-Proyecto Aprendizaje Profundo
### Autoras: Ana Blasco & Adriana Marí

# Asistente Multimodal de Salud para Personas Mayores

## 1. Descripción del proyecto

El objetivo de este proyecto es desarrollar un asistente inteligente que ayude a personas mayores a gestionar aspectos básicos de su salud y bienestar diario de forma sencilla. Muchas personas mayores tienen dificultades para recordar correctamente indicaciones médicas, como la posología de sus medicaciones o para leer documentos con letra pequeña, como los prospectos. Este sistema pretende facilitar dichas tares mediante el uso de inteligencia artificial y diferentes modalidades de entrada y salida de información.

El asistente permitirá interactuar con el usuario principalmente a través de la voz, aunque también podrá procesar otros tipos de entrada, como imágenes o documentos. Por ejemplo, una persona podrá pedir por voz qué medicamentos debe tomar ese día, o hacer una foto de una prescripción médica para que el sistema la analice y le lea las instrucciones de forma clara y comprensible. De esta manera, el sistema recibe información médica potencialmente compleja y la adaptada a voluntad del usuario.

El sistema se basa en un *pipeline* de modelos de inteligencia artificial preentrenados que permiten procesar diferentes tipos de datos. En primer lugar, si la entrada es audio, se utiliza un modelo de speech-to-text (S2T) para convertir la voz del usuario en texto y que un modelo de lenguaje (text-to-text, T2T) la analice. Si la entrada es una imagen o documento, se utilizan técnicas de lenguaje visual (VL) para extraer la información relevante en forma de texto.

Finalmente, el sistema genera una respuesta que puede presentarse de diferentes maneras. Por un lado, todas las respuestas se proporcionan en forma de voz utilizando un modelo de text-to-speech (T2S) para facilitar la comprensión y algunas también proporcionan una respuesta parcial escrita, acorde a la hablada. Por otro lado, también se pueden generar salidas visuales, como es el resumen de medicación del día, que es un recordatorio visual de los medicamentos que el usuario debe tomar ese día con sus respectivas dosis.

## 2. Diagrama de Flujo

El astistente es multimodal porque, com habíamos dicho anteriormente, acepta diferentes tipos de entrada: voz, imágenes o documentos PDF. Dependiendo del tipo de entrada, se utilizan distintos modelos de inteligencia artificial para convertir la información a texto. Después, un modelo de lenguaje interpreta la petición del usuario y genera una respuesta que puede presentarse en diferentes formatos, como audio, texto o tabla HTML para facilitar la comprensión.

<img src="https://github.com/ablasve/Mini-Proyecto-Asistente-Multimodal-de-Salud/blob/main/imagenes/estructura.png?raw=1" alt="Diagrama de flujo">

## 3. Fases del desarrollo del asistente

Decidimos separar el desarrollo del asistente multimodal en dos etapas diferenciadas: la fase A y la fase B. Por un lado, el objetivo principal que motiva la fase A es asentar la lógica de la interacción asitente-usuario, decidiendo como mostrar y acceder a las opciones que el asistente ofrece. Por otro lado, la fase B se focaliza en las funciones concretas que llevarán a cabo la ya mencionada interacción; aquellas funciones que recabarán la información e implementarán los modelos preentrenados. Aunque en este documento se expliquen ambas fases, remitimos al lector a los archivos `FaseA.ipynb` y `FaseB.ipynb` para más información y detalles.

En este documento no se incluirán las definiciones de las funciones empleadas sino diagramas de flujo para representar su estructura. Las funciones definidas se encuentran en el archivo `funciones_salud.py`.

Para observar el funcionamiento del asistente, se remite al archivo `Prueba_Asistente.ipynb` cuya guía de uso se encuentra disponible en el `README.md` del [repositorio](https://github.com/ablasve/Mini-Proyecto-Asistente-Multimodal-de-Salud.git ).

### 3.1. Fase A: gestión del asistente

Lo primero que fijamos en esta fase fue la forma en la que se guardaría la información del usuario: la _memoria_ del asistente es un archivo en formato _JSON_ que contiene un diccionario con tres campos: el nombre por el que dirigirse al usuario, el historial de medicaciones aportado y una lista indicando cuáles han sido las últimas medicinas incorporadas al historial.

Después de esto creamos una función de presentación para que, si existe el archivo de memoria se salude por su nombre al usuario y se pase directamente al menú principal. En caso contrario el asistente se presenta y le pide al usuario que le indique oralmente un nombre por que poder referirse a él/ella. Cabe recalcar que, a excepción de las imágenes o documentos PDF, decidimos que todas las entradas por parte del usuario fueran en audio, de forma que salvamos dos obstáculos: las dificultades que muchas veces presentan las personas mayores para leer y/o escribir en un teclado, y la accesibilidad del asistente, ya que de esta forma la interacción resulta en un diálogo. Para posibilitar dicho diálogo, elegimos el modelo Whisper AI para pasar el audio del usuario a texto, el modelo QWen2.5-3B-instruct para procesar el texto y el modelo Edge TTS para brindarle voz al asistente. Como podemos ver, a pesar de que en esta fase todavía no entramos de lleno en el uso de modelos multimodales con el objetivo que perseguimos, sí que son una pieza clave.

Otro punto que abordamos en esta fase fue la creación dos menús: el principal y el de ajustes. Desde el menú principal, a través del cual se puede seleccionar los distintos recursos desarrollados en la fase B y/o acabar la sesión, también se puede acceder al humilde menú de opciones, donde el usuario podrá elegir entre cambiar el nombre, borrar todo su historial de medicinas o volver al menú principal.

Por último, definimos la función global que unía todos los elementos indicados hasta ahora, siguiendo el esquema general que muestra la Imagen 1.

![Imagen 1. Estructura de la gestión del asistente](https://github.com/ablasve/Mini-Proyecto-Asistente-Multimodal-de-Salud/blob/main/'imagenes/estructura.png'?raw=1)


### 3.2. Fase B: Procesamiento Multimodal de la Información

Esta fase, la más extensa, se centra en definir las fuciones necesarias para abordar las tareas que tiene como objetivo el asistente. Recordemos que dichas tareas eran:

- Registrar un medicamento nuevo en el historial del a través de imágenes.
- Modificar o eliminar un medicamento del historial a través de la voz.
- Resumir la medicación de ese día mediante voz y una tabla HTML.
- Responder preguntas sencillas del usuario a través de la voz.
- Leer documentos PDF o imágenes en voz alta.

Por tanto, decidimos crear una función para cada opción, con las cuales integraremos cada funcionalidad en el menú principal de la fase A. A continuación explicamos detalladamente cada una.


#### 3.2.1. Registrar un medicamento nuevo en el historial a través de imágenes y voz

Para registrar un medicamento nuevo, creamos la función principal `subir_receta()`. Esta función es la que dirige todo el proceso: solicita al usuario que suba la imagen, llama a los modelos de inteligencia artificial y, finalmente, confirma si la acción se ha llevado a cabo correctamente. Para ello, se apoya en funciones auxiliares encargadas de tareas específicas que definiremos a continuación.

La primera de ellas es `analizar_receta(ruta_imagen, memoria, model_vision, processor_vision)`. Esta es la encargada de procesar la imagen de la receta médica o caja de pastillas proporcionada por el usuario. Para ello, implementamos el modelo de lenguaje visual (VL) **Qwen2-VL-2B-Instruct**. Elegimos este modelo por su capacidad para realizar reconocimiento óptico de caracteres (OCR) en imágenes y estructurar la información extraída siguiendo un *prompt* específico. En los parámetros de llamada al procesador visual, definimos unas restricciones de resolución (`min_pixels` y `max_pixels`) suficientemente altas para garantizar que el modelo pudiera leer la letra relativamente pequeña de los envases médicos, pero evitando pasarle imágenes demasiado grandes que saturaran nuestra memoria de procesamiento. El objetivo de este *prompt* es que el modelo devuelva un archivo en formato JSON con el nombre, la dosis y la fecha de finalización del tratamiento, ignorando datos irrelevantes.

En esta fase, añadimos una capa de seguridad, ya que si la imagen analizada es una caja de medicación que no contiene instrucciones de toma (la dosis o la fecha están vacíos en el JSON), el asistente detecta esta ausencia de datos en esos campos, y se abre el micrófono para preguntar directamente al paciente dicha información. El audio se transcribe con **Whisper** y se procesa con nuestro modelo de texto puro (**Qwen2.5-3B-Instruct**) para extraer la respuesta e integrarla en el archivo JSON.



graph TD
    A([Inicio: Opción 1]) --> B[Solicitar Imagen o PDF]
    B --> C{¿Archivo subido?}
    
    C -- Sí --> D[Llamada a <b>analizar_receta</b>]
    C -- No --> Z([Retornar memoria sin cambios])

    subgraph Fase de Visión Artificial
        D --> E((Modelo: Qwen2-VL-2B-Instruct))
        E --> F[Salida: JSON preliminar <br> nombre, dosis, fin]
    end

    F --> G{¿Falta dosis o fecha? <br> Ej. Es una caja}

    subgraph Fase de Interacción por Voz RAG
        G -- Sí --> H[<b>generar_voz:</b> Preguntar detalles faltantes]
        H --> I[<b>grabar_audio:</b> Escuchar al paciente]
        I --> J((Modelo STT: Whisper))
        J --> K[Texto transcrito]
        K --> L((Modelo Texto: Qwen2.5-3B-Instruct))
        L --> M[Salida: JSON con los datos faltantes]
        M --> N[Fusionar datos visuales y de voz]
    end

    G -- No --> N

    subgraph Fase de Registro y Confirmación
        N --> O[Llamada a <b>registrar_en_memoria</b>]
        O --> P[(memoria_salud.json)]
        O --> Q[<b>generar_voz:</b> Confirmar operación]
        Q --> R[<b>mostrar_recordatorios:</b> Tabla HTML]
        R --> S([Fin: Retornar memoria actualizada])
    end

# **REVISAR**

Una vez extraídos los datos, entra en juego la función `registrar_en_memoria(nuevos_datos)`. Esta función compara la nueva información con el historial existente en el archivo `memoria_salud.json` del usuario, evitando así registrar medicamentos duplicados, y actualiza la lista de últimas adiciones para poder ofrecer una confirmación oral precisa.

#### 3.2.2. Modificar o eliminar un medicamento del historial a través de la voz

Para mantener el historial médico actualizado, desarrollamos la función `cambios_meds_menu()`, la cual actúa como un submenú guiado por voz. Al entrar en esta opción, el sistema enuncia los medicamentos actuales y pregunta al usuario si desea "Eliminar" (opción 1) o "Modificar" (opción 2).

Para captar la respuesta del usuario, utilizamos el modelo **OpenAI Whisper** (versión `small`), elegido por su excelente precisión en la transcripción de audio en español. Al instanciar el modelo mediante la función `transcribe`, decidimos inyectar explícitamente dos parámetros cruciales para evitar "alucinaciones" (texto inventado por la IA en momentos de silencio):
* `condition_on_previous_text=False`: Evita que el modelo se quede atrapado en bucles repitiendo lo que ha escuchado anteriormente.
* `temperature=0.0`: Convierte el modelo en determinista, eliminando la "creatividad" y forzando una transcripción puramente literal.

*(Aquí podéis insertar el diagrama de cómo Whisper captura la voz y Qwen la normaliza)*

Dado que la transcripción en bruto ("quiero la opción uno", "borrar el paracetamol") debe convertirse en una instrucción lógica para el código (un dígito entero), empleamos nuevamente el modelo **Qwen2.5-3B-Instruct**. Este LLM recibe un *prompt* diseñado para clasificar la intención del usuario y normalizarla, extrayendo exclusivamente el número de la opción deseada. Este mismo paradigma de "transcripción por Whisper + extracción lógica por Qwen" se repite dentro de las funciones dependientes `eliminar_med()` y `modificar_med()`, permitiendo al paciente elegir por voz qué campo desea alterar (nombre, dosis o fecha) y validando la acción con un paso de confirmación ("Sí"/"No").

*(Aquí podéis insertar el bloque de código de `cambios_meds_menu`, `eliminar_med` y `modificar_med`)*

#### 3.2.3. Tabla resumen de tratamientos (Organización Visual)

A pesar de que el núcleo del asistente es vocal, consideramos vital ofrecer un apoyo visual para la gestión diaria de las pastillas. Para ello, creamos la función `resumen_visual(datos_medicinas)`.
Esta función extrae las medicinas de la memoria, filtra aquellas cuya fecha de finalización ya ha expirado (evitando mostrar medicación caducada) y aplica expresiones regulares (`re.split`) para limpiar el texto de las dosis. Mediante una lógica condicional basada en palabras clave (ej. "8 horas", "3 veces", "desayuno"), la función clasifica cada fármaco en las franjas de Mañana, Mediodía y Noche. Finalmente, la función no devuelve texto, sino que renderiza dinámicamente una tabla en formato HTML usando la librería `IPython.display`, mostrando al usuario de Google Colab una interfaz limpia, coloreada y altamente legible.

*(Aquí podéis insertar el bloque de código de `resumen_visual`)*

#### 3.2.4. Resolución de dudas médicas mediante voz (RAG)

Una de las características más avanzadas del asistente es su capacidad para actuar como consultor sobre la propia medicación del usuario mediante la función `preguntas()`.
El proceso funciona aplicando una arquitectura simplificada de Generación Aumentada por Recuperación (RAG). Primero, se transcribe la duda del usuario (ej. *"¿Puedo tomar ibuprofeno si ya me he tomado mis pastillas?"*). Acto seguido, la función carga la memoria local del usuario (`memoria.get("medicinas")`) y la convierte en texto plano.

Toda esta información (la pregunta y el contexto médico del paciente) se empaqueta en el *prompt* que se envía al modelo **Qwen2.5-3B-Instruct**. Las reglas del *prompt* son estrictas: el modelo debe responder de forma extremadamente breve, referirse *únicamente* al historial proporcionado y, ante cualquier duda médica de gravedad o interacciones no documentadas, derivar al paciente a su médico o farmacéutico. De esta forma, limitamos los riesgos inherentes de la IA en el ámbito de la salud y garantizamos que la respuesta locutada sea directa y segura.

*(Aquí podéis insertar el bloque de código de `preguntas`)*

#### 3.2.5. Lector automático de documentos e informes (PDF e Imágenes)

Para abordar la dificultad que tienen muchas personas mayores al leer informes médicos en formato PDF o con letra pequeña, desarrollamos la función `lector_docs()`.

Esta herramienta permite subir un archivo y utiliza la función auxiliar `analizar_documento()` para procesarlo. Si el archivo es un PDF, integramos la librería `pdf2image` para rasterizar el documento y convertir cada una de sus páginas en imágenes temporales PNG. A continuación, iteramos sobre cada página y utilizamos nuevamente nuestro modelo multimodal **Qwen2-VL-2B-Instruct**, pasándole como instrucción extraer y transcribir toda la información al español sin realizar adiciones.

Al finalizar, las imágenes temporales son eliminadas automáticamente del entorno de Colab por eficiencia, y el texto completo se consolida para ser leído en voz alta por el asistente mediante la herramienta `edge-tts`.

*(Aquí podéis insertar el bloque de código del lector de documentos)*